# NB4: Annotation-Free Difficulty Oracle
**AbstractAttentionKernel vs MLP baseline on COCO val2017**

Pre-registered target: AUROC > 0.70 (MLP baseline: 0.560)

In [25]:
!pip install openai-clip scikit-learn scipy -q

In [4]:
import sys, os
from pathlib import Path

# Auto-discover nb4_lar_graph.py
for search_root in ['/kaggle/input', '/kaggle/working']:
    for root, dirs, files in os.walk(search_root):
        if 'nb4_lar_graph.py' in files:
            sys.path.insert(0, root)
            print(f'Found nb4_lar_graph.py at: {root}')
            break

# Auto-discover COCO paths
IMAGES_DIR = None
CAPTIONS_JSON = None

for root, dirs, files in os.walk('/kaggle/input'):
    if Path(root).name == 'val2017' and any(f.endswith('.jpg') for f in files[:5]):
        IMAGES_DIR = root
    if 'captions_val2017.json' in files:
        CAPTIONS_JSON = os.path.join(root, 'captions_val2017.json')
    if IMAGES_DIR and CAPTIONS_JSON:
        break

print(f'Images dir:    {IMAGES_DIR}')
print(f'Captions file: {CAPTIONS_JSON}')
print(f'Images dir exists:    {os.path.isdir(IMAGES_DIR) if IMAGES_DIR else False}')
print(f'Captions file exists: {os.path.isfile(CAPTIONS_JSON) if CAPTIONS_JSON else False}')

if not IMAGES_DIR or not CAPTIONS_JSON:
    print('\n--- /kaggle/input tree (top 3 levels) ---')
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        if level < 3:
            print('  ' * level + os.path.basename(root) + '/')
        dirs[:] = [d for d in dirs if level < 3]

Found nb4_lar_graph.py at: /kaggle/input/datasets/aadithyavishnu/nb4-lar-graph
Images dir:    /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017
Captions file: /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/captions_val2017.json
Images dir exists:    True
Captions file exists: True


In [5]:
import nb4_lar_graph as nb4

pairs = nb4.load_coco_pairs(
    images_dir=IMAGES_DIR,
    captions_json=CAPTIONS_JSON,
    n=5000,
    seed=42
)
print(f'Loaded {len(pairs)} pairs')

[WARN] lar package not found — running in standalone mode (ABC contracts enforced manually)
[WARN] core/interfaces.py not found — ABC contracts defined inline
[NB4] Device: cuda
[NB4] Lár available: False | ABCs available: False
[load_coco_pairs] Loaded 5000 pairs (skipped 0)
Loaded 5000 pairs


In [11]:
import clip as _clip
from PIL import Image as PILImage

def _fixed_encode(self, x):
    self._load()
    if self._model is None:
        z = torch.randn(self.output_dim, device=nb4.DEVICE)
        return z / z.norm()
    with torch.no_grad():
        if isinstance(x, PILImage.Image):
            img_tensor = self._prep(x).unsqueeze(0).to(nb4.DEVICE)
            return self._model.encode_image(img_tensor).squeeze(0).float()
        elif hasattr(x, 'shape'):
            return self._model.encode_image(x.to(nb4.DEVICE)).float()
        else:
            return self._model.encode_text(
                _clip.tokenize([x]).to(nb4.DEVICE)
            ).squeeze(0).float()

nb4.CLIPEncoderNode.encode = _fixed_encode

In [9]:
!pip install openai-clip scikit-learn scipy -q

In [12]:
import clip as _clip
from PIL import Image as PILImage
import torch, torch.nn.functional as F

# ── 1. encode: handle PIL images correctly ──────────────────
def _fixed_encode(self, x):
    self._load()
    if self._model is None:
        z = torch.randn(self.output_dim, device=nb4.DEVICE)
        return z / z.norm()
    with torch.no_grad():
        if isinstance(x, PILImage.Image):
            img_tensor = self._prep(x).unsqueeze(0).to(nb4.DEVICE)
            return self._model.encode_image(img_tensor).squeeze(0).float()
        elif hasattr(x, 'shape'):
            return self._model.encode_image(x.to(nb4.DEVICE)).float()
        else:
            return self._model.encode_text(
                _clip.tokenize([x]).to(nb4.DEVICE)
            ).squeeze(0).float()
nb4.CLIPEncoderNode.encode = _fixed_encode

# ── 2. D-score: semantic cosine distance, calibrated tau ────
def _cosine_d_execute(self, state):
    z_imgs = state.get('z_imgs')
    z_txts = state.get('z_txts')
    z_i = F.normalize(z_imgs, dim=-1)
    z_t = F.normalize(z_txts, dim=-1)
    d_scores = 1 - (z_i * z_t).sum(dim=-1)
    self.tau = torch.quantile(d_scores, 0.70).item()
    labels = (d_scores >= self.tau).float()
    state.set('d_scores', d_scores)
    state.set('labels', labels)
    state.set('calibrated_tau', self.tau)
    n_hard = labels.sum().int().item()
    print(f'[D_ScoreLabeler] Cosine D-scores (V1-V6, τ={self.tau:.4f})...')
    print(f'  D-scores: min={d_scores.min():.3f} max={d_scores.max():.3f} | Hard cases: {n_hard}/{len(d_scores)} ({100*n_hard/len(d_scores):.1f}%)')
    return self.next_node
nb4.D_ScoreLabeler.execute = _cosine_d_execute

# ── 3. oracle: random eval split so Hard > 0 ────────────────
def _fixed_oracle_execute(self, state):
    hcl_K    = state.get('hcl_K')
    hcl_V    = state.get('hcl_V')
    z_imgs   = state.get('z_imgs')
    d_scores = state.get('d_scores')
    tau      = state.get('calibrated_tau', 0.3)
    n = len(z_imgs)
    seed = state.get('seed', 42)
    perm = torch.randperm(n, generator=torch.Generator().manual_seed(seed))
    eval_idx = perm[:n // 5]
    z_eval   = z_imgs[eval_idx]
    d_eval   = d_scores[eval_idx]
    y_true   = (d_eval >= tau).cpu().numpy().astype(int)
    y_score_full = self._predict_full(z_eval, hcl_K, hcl_V)
    y_score_knn  = self._predict_knn(z_eval, hcl_K, hcl_V)
    state.set('oracle_scores_full', y_score_full)
    state.set('oracle_scores_knn',  y_score_knn)
    state.set('eval_labels',        y_true)
    state.set('eval_idx',           eval_idx)
    print(f'[HCLAttentionOracle] Running attention oracle (I1-I6, k={self.k})...')
    print(f'  Eval set: {len(eval_idx)} pairs | Hard: {y_true.sum()} | Easy: {(1-y_true).sum()}')
    return self.next_node
nb4.HCLAttentionOracle.execute = _fixed_oracle_execute

# ── 4. write paths: /kaggle/working not read-only input ─────
def _patched_checkpoint_init(self, checkpoint_path='/kaggle/working/nb4_checkpoint.json', next_node=None):
    self.path = checkpoint_path
    self.next_node = next_node
nb4.ResumeCheckpointNode.__init__ = _patched_checkpoint_init

def _patched_logger_init(self, secret=nb4.HMAC_SECRET, log_dir='/kaggle/working/nb4_lar_logs', next_node=None):
    self.secret = secret
    self.log_dir = log_dir
    self.next_node = next_node
nb4.ResultLoggerNode.__init__ = _patched_logger_init

In [13]:
import torch, numpy as np
from nb4_lar_graph import build_nb4_graph, GraphState

def run_nb4_coco(pairs, seed=42, k=10):
    torch.manual_seed(seed)
    np.random.seed(seed)
    print(f'\n{"="*60}')
    print(f'  NB4  |  seed={seed}  k={k}  n={len(pairs)}')
    print(f'{"="*60}\n')
    entry, executor = build_nb4_graph(k=k)
    state = GraphState({'raw_pairs': pairs, 'n_pairs': len(pairs), 'seed': seed})
    node = entry
    while node is not None:
        node = node.execute(state)
    return dict(state)

result = run_nb4_coco(pairs, seed=42, k=10)


  NB4  |  seed=42  k=10  n=5000

[CLIPEncoderNode] Encoding pairs (M1-M3)...


100%|███████████████████████████████████████| 338M/338M [00:05<00:00, 61.2MiB/s]


  Encoded 5000 pairs → z_imgs: torch.Size([5000, 512])
[D_ScoreLabeler] Cosine D-scores (V1-V6, τ=0.7136)...
  D-scores: min=0.575 max=0.868 | Hard cases: 1500/5000 (30.0%)
[HCL_BuilderNode] Building HCL (top-80% hard pairs)...
  HCL size: 4000 pairs | K: torch.Size([4000, 512]) | V: torch.Size([4000])
[EncoderGenerationNode] Re-encoding HCL raw pairs (P1-P6 zombie-action defence)...
  Encoder Δ (mean L2): 0.000000  (null update — P3 satisfied)
[InvariantCheckerNode] Verifying A1-A6 kernel invariants...
  ✅ A1 compute() signature
  ✅ A2 weights.shape == K.shape[0]
  ✅ A3 weights >= 0
  ✅ A4 sum(weights) ≈ 1
  ✅ A5 topk descending
  ✅ A6 len(topk) == k
  All A1-A6 invariants satisfied — oracle cleared to run
[HCLAttentionOracle] Running attention oracle (I1-I6, k=10)...
  Eval set: 1000 pairs | Hard: 297 | Easy: 703
[MLP_BaselineNode] Running MLP baseline (I2 violated — Exp 3 replicated)...
  MLP scores: min=0.020 max=0.928
[AUROC_JudgeNode] Computing AUROC (R1-R4, A/B Tester fan-in)...

In [14]:
from scipy import stats

SEEDS = [42, 7, 13, 99, 2025]
oracle_aurocs, mlp_aurocs = [], []

for seed in SEEDS:
    r = run_nb4_coco(pairs, seed=seed, k=10)
    if r.get('auroc_oracle_knn') is not None:
        oracle_aurocs.append(r['auroc_oracle_knn'])
        mlp_aurocs.append(r.get('auroc_mlp', 0.560))

if oracle_aurocs:
    t, p = stats.ttest_rel(oracle_aurocs, mlp_aurocs)
    print(f'\n{"="*60}')
    print(f'  NB4 5-SEED FINAL RESULTS')
    print(f'{"="*60}')
    print(f'  Oracle AUROC:  {np.mean(oracle_aurocs):.4f} ± {np.std(oracle_aurocs):.4f}')
    print(f'  MLP AUROC:     {np.mean(mlp_aurocs):.4f} ± {np.std(mlp_aurocs):.4f}')
    print(f'  Δ AUROC:       {np.mean(oracle_aurocs)-np.mean(mlp_aurocs):+.4f}')
    print(f'  Paired t-test: t={t:.3f}  p={p:.4f}')
    verdict = 'CONFIRMED ✅' if np.mean(oracle_aurocs) > 0.70 else 'NOT CONFIRMED ❌'
    print(f'  Verdict:       {verdict}')
    print(f'{"="*60}')


  NB4  |  seed=42  k=10  n=5000

[CLIPEncoderNode] Encoding pairs (M1-M3)...
  Encoded 5000 pairs → z_imgs: torch.Size([5000, 512])
[D_ScoreLabeler] Cosine D-scores (V1-V6, τ=0.7136)...
  D-scores: min=0.575 max=0.868 | Hard cases: 1500/5000 (30.0%)
[HCL_BuilderNode] Building HCL (top-80% hard pairs)...
  HCL size: 4000 pairs | K: torch.Size([4000, 512]) | V: torch.Size([4000])
[EncoderGenerationNode] Re-encoding HCL raw pairs (P1-P6 zombie-action defence)...
  Encoder Δ (mean L2): 0.000000  (null update — P3 satisfied)
[InvariantCheckerNode] Verifying A1-A6 kernel invariants...
  ✅ A1 compute() signature
  ✅ A2 weights.shape == K.shape[0]
  ✅ A3 weights >= 0
  ✅ A4 sum(weights) ≈ 1
  ✅ A5 topk descending
  ✅ A6 len(topk) == k
  All A1-A6 invariants satisfied — oracle cleared to run
[HCLAttentionOracle] Running attention oracle (I1-I6, k=10)...
  Eval set: 1000 pairs | Hard: 297 | Easy: 703
[MLP_BaselineNode] Running MLP baseline (I2 violated — Exp 3 replicated)...
  MLP scores: min=0